<a href="https://colab.research.google.com/github/aaradhyajain1110/Assignments-celebal-internship/blob/main/Copy_of_Vigilant_Multimodal_Bullying_SCHOOL%7C_INCOGNITO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
import os

print("Downloading dataset...")
# This downloads the dataset straight to Colab's temporary storage
dataset_path = kagglehub.dataset_download('mohamedmustafa/real-life-violence-situations-dataset')
base_dir = os.path.join(dataset_path, "Real Life Violence Dataset")

print(f"Dataset successfully downloaded to: {base_dir}")

Using Colab cache for faster access to the 'real-life-violence-situations-dataset' dataset.
Dataset successfully downloaded to: /kaggle/input/real-life-violence-situations-dataset/Real Life Violence Dataset


In [ ]:
import cv2
import glob
from tqdm import tqdm

output_dir = "/content/image_dataset"
os.makedirs(os.path.join(output_dir, "Violence"), exist_ok=True)
os.makedirs(os.path.join(output_dir, "NonViolence"), exist_ok=True)

def extract_frames(video_path, class_name, max_frames=15):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    skip = max(1, total_frames // max_frames)

    count = 0
    frame_id = 0
    while cap.isOpened() and count < max_frames:
        ret, frame = cap.read()
        if not ret: break
        if frame_id % skip == 0:
            img_name = f"{class_name}_{os.path.basename(video_path)}_{count}.jpg"
            cv2.imwrite(os.path.join(output_dir, class_name, img_name), frame)
            count += 1
        frame_id += 1
    cap.release()

print("Extracting frames from videos... This will take a minute or two.")
for class_name in ["Violence", "NonViolence"]:
    # We grab 100 videos per class to keep the dataset light and training fast
    videos = glob.glob(f"{base_dir}/{class_name}/*.mp4")[:100]
    for vid in tqdm(videos, desc=class_name):
        extract_frames(vid, class_name)

print("Frame extraction complete! Your images are ready.")

Extracting frames from videos... This will take a minute or two.


NonViolence: 100%|██████████| 100/100 [00:11<00:00,  8.65it/s]

Frame extraction complete! Your images are ready.


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

# 1. Load the extracted images
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_gen = datagen.flow_from_directory(
    '/content/image_dataset', target_size=(224, 224),
    batch_size=32, class_mode='binary', subset='training'
)
val_gen = datagen.flow_from_directory(
    '/content/image_dataset', target_size=(224, 224),
    batch_size=32, class_mode='binary', subset='validation'
)

# 2. Build the Model
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False # Freeze the base brain so we don't break it

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.2)(x) # Prevents memorization
predictions = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 3. Train the Model
print("Starting training...")
model.fit(train_gen, epochs=5, validation_data=val_gen)

# 4. The Magic Step: Convert to TFLite for Raspberry Pi
print("Converting model to TensorFlow Lite format...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT] # This shrinks it for the Pi's 1GB RAM
tflite_model = converter.convert()

# Save the file
with open('/content/bullying_model.tflite', 'wb') as f:
    f.write(tflite_model)

print("SUCCESS! Your model is saved as bullying_model.tflite")

Found 2400 images belonging to 2 classes.
Found 600 images belonging to 2 classes.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Starting training...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 43s 328ms/step - accuracy: 0.6572 - loss: 0.6397 - val_accuracy: 0.8150 - val_loss: 0.3710
Epoch 2/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 8s 109ms/step - accuracy: 0.9012 - loss: 0.2617 - val_accuracy: 0.8083 - val_loss: 0.3505
Epoch 3/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 8s 113ms/step - accuracy: 0.9278 - loss: 0.1917 - val_accuracy: 0.8617 - val_loss: 0.3045
Epoch 4/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 8s 113ms/step - accuracy: 0.9578 - loss: 0.1408 - val_accuracy: 0.8200 - val_loss: 0.3805
Epoch 5/5
75/75 ━━━━━━━━━━━━━━━━━━━━ 9s 125ms/step - accuracy: 0.9554 - loss: 0.1320 - val_accuracy: 0.8467 - val_loss: 0.3470
Converting model to TensorFlow Lite format...
Saved artifact at '/tmp/tmp15movpw9'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  132574773557200: TensorSpec(shap